# Go2 RL 보행 학습 (Colab)
1. 셀 1~4: GPU 확인, Drive 마운트, 설치, **T4 속도 측정**(공식 Genesis Go2 예제, 이 저장소와 무관)
2. 셀 5: 이 저장소 클론 후 학습. Drive에 체크포인트를 저장하고 재연결 시 `--resume`

> 셀 5는 RL 코드(`module1_locomotion/rl_training`)가 GitHub에 push되어 있어야 한다. 아래 `BRANCH`가 그 브랜치를 가리켜야 한다.

In [ ]:
BRANCH = "main"  # RL 코드가 push된 브랜치(학습 셀에서 사용)

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 속도 측정은 push 전에도 돌 수 있게 핀을 직접 설치(Colab 기본 torch 사용)
!pip install -q genesis-world==1.5.0 rsl-rl-lib==6.0.0 tensorboard

## 속도 측정 (T4)
Genesis v1.5.0의 공식 `go2_train.py`로 101 iter를 돌려 총 시간과 step/s를 본다. 비교 기준: Mac 약 5.7만 step/s(공식 예제). 이 저장소 env는 물리 4 substep이라 Mac에서 3.1만 step/s였다.

In [ ]:
import os, subprocess
BASE = "https://raw.githubusercontent.com/Genesis-Embodied-AI/genesis-world/v1.5.0/examples/locomotion"
os.makedirs("/content/speed", exist_ok=True)
for f in ("go2_env.py", "go2_train.py"):
    subprocess.run(["curl", "-fsSL", "-o", f"/content/speed/{f}", f"{BASE}/{f}"], check=True)
    assert os.path.getsize(f"/content/speed/{f}") > 0, f
r = subprocess.run(
    "cd /content/speed && python go2_train.py -e t4_speed -b 4096 --max-iterations 101",
    shell=True, capture_output=True, text=True)
out = r.stdout + r.stderr
print("returncode:", r.returncode)
print(out[-1500:])
for key in ("steps per second", "time elapsed"):
    hits = [l.strip() for l in out.splitlines() if key in l.lower()]
    print("(none)" if not hits else hits[-1])  # 마지막 줄 = 101 iter 누적

## 학습
첫 셀의 `BRANCH`가 RL 코드가 있는 브랜치여야 한다. `MAX_ITER`는 총 반복 수이며 재연결 시 남은 만큼만 학습한다. 로그·체크포인트는 Drive `rl_go2/<EXP>`에 쌓이고, 이미 `model_*.pt`가 있으면 `--resume`으로 이어서 학습한다.

`TERRAIN = True`면 지형 커리큘럼(`--terrain`)으로 학습한다. 재연결해도 같은 값을 둔다 — `--resume`은 run 시작 때와 지형 모드·학습 코드가 다르면 거부한다.

In [ ]:
EXP = "rough_v1"
TERRAIN = True  # 지형 커리큘럼(--terrain). 평지 학습은 False
BATCH = 4096
MAX_ITER = 3000  # 총 반복 수(재연결해도 여기까지만 학습)
LOG_DIR = "/content/drive/MyDrive/rl_go2"  # train.py가 뒤에 <EXP>를 붙인다

In [ ]:
import glob, os, re
REPO = "/content/32-dt-doosan-quadruped"
if not os.path.isdir(REPO):
    !git clone -b $BRANCH https://github.com/codyssey-dt-doosan/32-dt-doosan-quadruped.git $REPO
%cd $REPO/module1_locomotion/rl_training
!grep -v '^torch==' requirements.txt > /content/req.txt
!pip install -q -r /content/req.txt
ckpts = [int(re.search(r"model_(\d+)\.pt", p).group(1)) for p in glob.glob(f"{LOG_DIR}/{EXP}/model_*.pt")]
done = max(ckpts, default=0)
remaining = MAX_ITER - done
resume = "--resume" if ckpts else ""
terrain = "--terrain" if TERRAIN else ""
print("done:", done, "remaining:", remaining, "resume:", bool(resume), "terrain:", TERRAIN)
if remaining > 0:
    !python train.py -e $EXP -b $BATCH --max-iterations $remaining --log-dir $LOG_DIR --save-interval 50 $resume $terrain